# Credit Card Fraud Detection - Exploratory Data Analysis

This notebook explores the Kaggle Credit Card Fraud Detection dataset containing 284,807 transactions. The goal of this EDA is to uncover patterns of fraudulent behavior, understand the extreme class imbalance, study temporal and transaction amount trends, and identify the predictive power of the PCA-reduced features.

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

# Load configuration
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

raw_path = os.path.join("..", config["data"]["raw_path"])
print(f"Raw dataset path: {raw_path}")

## 1. Load Data & Basic Structure

In [ ]:
if not os.path.exists(raw_path):
    print("Error: CSV file not found. Make sure you run 'python download_data.py' from the root directory first.")
else:
    df = pd.read_csv(raw_path)
    print(f"Dataset shape: {df.shape}")
    display(df.head())

## 2. Class Distribution Analysis

Let's look at the class imbalance. The target column `Class` has `1` for fraud and `0` for legitimate transactions.

In [ ]:
class_counts = df['Class'].value_counts()
class_pcts = df['Class'].value_counts(normalize=True) * 100

print("Class Distribution:")
print(pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_pcts}))

plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df, palette=['royalblue', 'crimson'])
plt.title('Transaction Class Distribution (Log Scale)')
plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.xticks([0, 1], ['Legit (0)', 'Fraud (1)'])
plt.show()

**Observations:**
* Out of 284,807 transactions, only 492 are labeled as fraud (0.17%).
* Because of this severe class imbalance, metrics like accuracy are highly misleading. If a model predicts all transactions as legitimate, it achieves 99.82% accuracy but catches 0% of fraud.
* We will need balancing techniques like SMOTE and threshold optimization targeting the F1-score/PR-AUC rather than accuracy.

## 3. Temporal Patterns: Time vs Class

The `Time` column is the number of seconds elapsed since the first transaction in the dataset. A full dataset covers exactly 2 days (48 hours = 172,800 seconds).
We can extract the hour of the day to see if fraud occurs more frequently during specific hours (e.g., late at night).

In [ ]:
# Convert time to hour of the day (0-23)
df['Hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Plot distribution of legit transactions
sns.histplot(data=df[df['Class'] == 0], x='Hour', bins=24, color='royalblue', kde=True, ax=axes[0])
axes[0].set_title('Legitimate Transactions Hourly Distribution')
axes[0].set_ylabel('Transaction Count')

# Plot distribution of fraud transactions
sns.histplot(data=df[df['Class'] == 1], x='Hour', bins=24, color='crimson', kde=True, ax=axes[1])
axes[1].set_title('Fraudulent Transactions Hourly Distribution')
axes[1].set_ylabel('Transaction Count')
axes[1].set_xlabel('Hour of the Day')

plt.tight_layout()
plt.show()

**Observations:**
* Legitimate transactions drop significantly between hours 1 to 6 AM, which corresponds to typical sleep cycles.
* Fraudulent transactions, however, remain relatively active or even peak during these early morning hours (hours 2 to 5 AM) when detection vigilance may be lower or automated triggers are less active.
* The temporal hour is indeed a useful feature for predicting fraud.

## 4. Transaction Amount Analysis

Let's look at the distribution of transaction amounts for legitimate and fraudulent classes.

In [ ]:
print("Legit Transaction Amount Summary:")
display(df[df['Class'] == 0]['Amount'].describe())

print("\nFraud Transaction Amount Summary:")
display(df[df['Class'] == 1]['Amount'].describe())

# Boxplot comparison
plt.figure(figsize=(8, 5))
sns.boxplot(x='Class', y='Amount', data=df, palette=['royalblue', 'crimson'])
plt.title('Transaction Amount Boxplot Comparison (Log Scale Y)')
plt.yscale('log')
plt.ylabel('Amount (Log Scale)')
plt.xticks([0, 1], ['Legit', 'Fraud'])
plt.show()

**Observations:**
* The average transaction amount for fraudulent transactions ($122.21) is slightly higher than for legitimate ones ($88.29).
* However, the maximum amount for fraud is $2,125.87, whereas legitimate transactions can go up to $25,691.16.
* Amount is highly right-skewed. Taking a log transform `np.log1p(Amount)` will help model stability.

## 5. PCA Visualization (Fraud vs Legit)

Columns V1 to V28 are the principal components obtained using PCA (due to confidentiality). Let's plot V17 and V14, which are known to be highly predictive, to visualize class separation.

In [ ]:
plt.figure(figsize=(10, 8))
# Plot legitimate in light gray/blue first to avoid drawing over fraud
sns.scatterplot(x='V17', y='V14', data=df[df['Class'] == 0], color='lightgray', alpha=0.3, label='Legit', s=10)
sns.scatterplot(x='V17', y='V14', data=df[df['Class'] == 1], color='crimson', alpha=0.9, label='Fraud', s=20)
plt.title('PCA Component Scatter: V17 vs V14')
plt.xlabel('V17')
plt.ylabel('V14')
plt.legend()
plt.show()

**Observations:**
* Fraudulent transactions cluster very distinctly towards negative values of both V17 and V14.
* Even though there is some overlap, the classes show a clear boundary, which makes XGBoost tree splits highly suitable for separating them.

## 6. Correlation Heatmap

Let's calculate correlation coefficients between all features and the target variable `Class` to see which PCA features are most informative.

In [ ]:
correlations = df.corr()['Class'].sort_values()
print("Top 5 Negatively Correlated Features:")
print(correlations.head(5))
print("\nTop 5 Positively Correlated Features:")
print(correlations.tail(6)[:-1]) # exclude Class correlation with itself

# Compute correlation matrix of V-features and class
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of All Features')
plt.show()

**Observations:**
* Since V1 to V28 are principal components, they have zero correlation with each other (orthogonal).
* Features `V17`, `V14`, `V12`, `V10`, `V16`, `V3`, `V7` show strong negative correlations with `Class` (meaning more negative values strongly imply fraud).
* Features `V11`, `V4`, `V2`, `V19` show positive correlations with `Class`.
* We can leverage interactions between these correlated features (e.g. `V17 * V14`) to boost model predictive power.

## Conclusion & Modeling Strategy

1. **Imbalance Handling:** SMOTE will be applied to the minority class in the training subset to prevent the model from ignoring the fraud cases.
2. **Feature Engineering:** We will add `Amount_Log`, `Hour_Of_Day`, and key component interactions like `V17_x_V14` and `V12_x_V10` to enhance decision boundaries.
3. **Tuning Metric:** We must maximize Average Precision (PR-AUC) during hyperparameter tuning rather than ROC-AUC or accuracy. In highly imbalanced datasets, PR-AUC focuses on the positive class performance, which directly represents fraud detection efficacy.
4. **F1 Optimization:** The decision threshold must be tuned using the precision-recall trade-off to match operational business constraints.